# ContainerClaw API Diagnostic
Use this notebook to verify your Gemini API key and the LLM Gateway connectivity.

## 1. Direct API Test (Host to Google)
This tests if your API key is valid and has remaining quota.

In [3]:
import requests
import os

# Read key from the project secrets
key_path = "../secrets/gemini_api_key.txt"
with open(key_path, "r") as f:
    GEMINI_API_KEY = f.read().strip()

model = "gemini-2.5-flash"
url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={GEMINI_API_KEY}"
payload = {"contents": [{"parts": [{"text": "Hi! Are you working?"}]}]}

print(f"Testing model: {model}...")
response = requests.post(url, json=payload)

if response.status_code == 200:
    print("✅ SUCCESS!")
    print("Response:", response.json()['candidates'][0]['content']['parts'][0]['text'])
else:
    print(f"❌ FAILED (Status {response.status_code})")
    print(response.text)

Testing model: gemini-2.5-flash...
✅ SUCCESS!
Response: Hi! As an AI, I don't have a concept of "working" in the human sense (like having a job or clocking in), but I am active and ready to process your requests and assist you right now!

So, you could say I'm always "on duty" when you interact with me. How can I help you today?


In [2]:
import requests

# Assuming GEMINI_API_KEY is already loaded
url = f"https://generativelanguage.googleapis.com/v1beta/models?key={GEMINI_API_KEY}"
response = requests.get(url)

if response.status_code == 200:
    models = response.json().get('models', [])
    for m in models:
        print(f"Name: {m['name']} | Supported Methods: {m.get('supportedGenerationMethods')}")

Name: models/gemini-2.5-flash | Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.5-pro | Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash | Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash-001 | Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash-lite-001 | Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash-lite | Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.5-flash-preview-tts | Supported Methods: ['countTokens', 'generateContent']
Name: models/gemini-2.5-pro-preview-tts | Supported Methods: ['countTokens', 'generateCo

## 2. Gateway Test (Host to Container)
This tests if the LLM Gateway container is routing correctly.

In [5]:
gateway_url = "http://localhost:8000/v1/chat/completions"
payload = {
    "model": "gemini-2.5-flash",
    "messages": [{"role": "user", "content": "Hello via the gateway!"}]
}

print("Testing Gateway routing...")
try:
    response = requests.post(gateway_url, json=payload, timeout=10)
    if response.status_code == 200:
        print("✅ GATEWAY SUCCESS!")
        print(response.json())
    else:
        print(f"❌ GATEWAY FAILED (Status {response.status_code})")
        print(response.text)
except Exception as e:
    print(f"❌ CONNECTION ERROR: {e}")

Testing Gateway routing...
✅ GATEWAY SUCCESS!
{'candidates': [{'content': {'parts': [{'text': "Hello right back through the digital gateway!\n\nIt's wonderful to connect. How can I assist you today, or what's on your mind?"}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}], 'usageMetadata': {'promptTokenCount': 5, 'candidatesTokenCount': 31, 'totalTokenCount': 840, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 5}], 'thoughtsTokenCount': 804}, 'modelVersion': 'gemini-2.5-flash', 'responseId': 'n1u0aeqtIIydqtsPhMuSkQk'}
